# L13 — Multi-Stage Systems: The Clinic Running Case Study

**Module**: M05 | **Chapter**: 7 | **Lecture**: L13

## Learning Objectives
By the end of this notebook you will be able to:
1. Model a multi-stage service system where entities pass through sequential resources.
2. Implement stage-specific queues and routing in SimPy.
3. Identify the bottleneck stage analytically and verify via simulation.
4. Show that simple per-stage queueing formulas do **not** compose for multi-stage systems.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as t_dist
from dataclasses import dataclass, field

## 1. System Description: Three-Stage Clinic

Patients flow through three sequential stages:

```
Arrival → [Registration Queue] → Registration → [Triage Queue] → Triage Nurse → [Exam Queue] → Physician → Discharge
```

| Stage | Servers | Service rate (patients/hr) |
|---|---|---|
| Registration | 1 | μ₁ = 20 |
| Triage Nurse | c₂ (decision variable) | μ₂ = 7.5 |
| Physician Exam | 2 | μ₃ = 4 |

Patient arrival rate: λ = 5 patients/hr

**Study question**: With how many nurses (c₂) is the bottleneck shifted, and what is the minimum mean patient time in clinic?

In [ ]:
# Analytical throughput bound per stage
lam = 5.0   # patients/hr

stages = {
    'Registration': {'mu': 20.0, 'c': 1},
    'Triage':       {'mu':  7.5, 'c': 2},   # try c=1 and c=2
    'Physician':    {'mu':  4.0, 'c': 2},
}

print(f"λ = {lam} patients/hr")
print(f"{'Stage':15s}  {'c':>3s}  {'μ/server':>10s}  {'ρ':>8s}  {'Bottleneck?':>12s}")
print('-' * 60)
for name, s in stages.items():
    rho = lam / (s['c'] * s['mu'])
    print(f"{name:15s}  {s['c']:>3d}  {s['mu']:>10.1f}  {rho:>8.3f}  "
          f"{'*** BOTTLENECK ***' if rho == max(lam/(v['c']*v['mu']) for v in stages.values()) else ''}")

## 2. SimPy Implementation: Three-Stage Patient

In [ ]:
@dataclass
class ClinicParams:
    lam:      float = 5.0    # arrivals/hr
    mu_reg:   float = 20.0   # registration rate
    mu_nurse: float =  7.5   # nurse rate
    mu_exam:  float =  4.0   # physician rate
    c_reg:    int   = 1
    c_nurse:  int   = 2
    c_exam:   int   = 2
    sim_time: float = 10_000.0
    seed:     int   = 0


def run_clinic(params: ClinicParams) -> dict:
    """Simulate three-stage clinic; return per-stage and overall performance."""
    rng = np.random.default_rng(params.seed)
    env = simpy.Environment()

    reg   = simpy.Resource(env, capacity=params.c_reg)
    nurse = simpy.Resource(env, capacity=params.c_nurse)
    exam  = simpy.Resource(env, capacity=params.c_exam)

    records = []

    def patient():
        t_arrive = env.now

        # Stage 1: Registration
        with reg.request() as req:
            yield req
            t_reg_start = env.now
            yield env.timeout(rng.exponential(1.0 / params.mu_reg))
        t_reg_end = env.now

        # Stage 2: Triage Nurse
        with nurse.request() as req:
            yield req
            t_nurse_start = env.now
            yield env.timeout(rng.exponential(1.0 / params.mu_nurse))
        t_nurse_end = env.now

        # Stage 3: Physician Exam
        with exam.request() as req:
            yield req
            t_exam_start = env.now
            yield env.timeout(rng.exponential(1.0 / params.mu_exam))
        t_depart = env.now

        records.append({
            'arrive':      t_arrive,
            'depart':      t_depart,
            'sojourn':     t_depart - t_arrive,
            'wait_reg':    t_reg_start   - t_arrive,
            'wait_nurse':  t_nurse_start - t_reg_end,
            'wait_exam':   t_exam_start  - t_nurse_end,
        })

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / params.lam))
            env.process(patient())

    env.process(arrivals())
    env.run(until=params.sim_time)

    df = pd.DataFrame(records)
    return {
        'W':          df['sojourn'].mean(),
        'Wq_reg':     df['wait_reg'].mean(),
        'Wq_nurse':   df['wait_nurse'].mean(),
        'Wq_exam':    df['wait_exam'].mean(),
        'n_served':   len(df),
        'df':         df,
    }


p = ClinicParams()
result = run_clinic(p)
print(f"Overall W = {result['W']:.4f} hr = {result['W']*60:.1f} min")
print(f"Wait at Registration: {result['Wq_reg']*60:.2f} min")
print(f"Wait at Triage Nurse: {result['Wq_nurse']*60:.2f} min")
print(f"Wait at Physician:    {result['Wq_exam']*60:.2f} min")

## 3. Bottleneck Analysis: Vary the Number of Nurses

In [ ]:
rows = []
for c_nurse in [1, 2, 3, 4]:
    p = ClinicParams(c_nurse=c_nurse, sim_time=50_000)
    r = run_clinic(p)
    rows.append({
        'c_nurse': c_nurse,
        'ρ_nurse': lam / (c_nurse * 7.5),
        'Wq_nurse_min': r['Wq_nurse'] * 60,
        'Wq_exam_min':  r['Wq_exam']  * 60,
        'W_total_min':  r['W'] * 60,
    })

sweep_df = pd.DataFrame(rows)
print(sweep_df.to_string(index=False, float_format='{:.3f}'.format))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sweep_df['c_nurse'], sweep_df['W_total_min'],   'ko-', lw=2, label='Total W (min)')
ax.plot(sweep_df['c_nurse'], sweep_df['Wq_nurse_min'],  'bs--', lw=1.5, label='Wait for nurse (min)')
ax.plot(sweep_df['c_nurse'], sweep_df['Wq_exam_min'],   'r^--', lw=1.5, label='Wait for physician (min)')
ax.set_xlabel('Number of triage nurses')
ax.set_ylabel('Minutes')
ax.set_title('Bottleneck shift: adding nurses moves pressure to physician stage')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Why Stage-Independent Analysis Fails

You might think: compute W for each stage separately using M/M/c formulas and add them.
But departure from the registration stage is **not** a Poisson process — it is bursty when the registration server is saturated.

In [ ]:
from math import factorial

def erlang_c_W(c, lam, mu):
    rho = lam / (c * mu)
    if rho >= 1.0:
        return float('inf')
    a = lam / mu
    s = sum(a**n / factorial(n) for n in range(c))
    last = (a**c / factorial(c)) / (1 - rho)
    C = last / (s + last)
    Wq = C / (c * mu - lam)
    return Wq + 1.0 / mu

# Independent-stage approximation (assumes Poisson inter-stage flow)
p_default = ClinicParams()
W_ind = (
    erlang_c_W(p_default.c_reg,   lam, p_default.mu_reg)   +
    erlang_c_W(p_default.c_nurse, lam, p_default.mu_nurse) +
    erlang_c_W(p_default.c_exam,  lam, p_default.mu_exam)
)

W_sim = run_clinic(ClinicParams(sim_time=200_000))['W']

print(f"Independent-stage sum of W:  {W_ind*60:.2f} min")
print(f"Simulation W (T=200,000 hr): {W_sim*60:.2f} min")
print(f"Approximation error: {abs(W_ind - W_sim)/W_sim*100:.1f}%")
print()
print("The approximation overestimates because inter-stage flow is NOT Poisson.")
print("A bursty arrival stream to stage 2 is worse than Poisson at the same rate.")

## 5. Sojourn Time Distribution

The distribution of total time in clinic is right-skewed — a useful summary for communication.

In [ ]:
result_large = run_clinic(ClinicParams(sim_time=500_000, c_nurse=2))
sojourns_min = result_large['df']['sojourn'] * 60

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(sojourns_min, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(sojourns_min.mean(),   color='red',    lw=2, label=f'Mean = {sojourns_min.mean():.1f} min')
axes[0].axvline(sojourns_min.median(), color='orange', lw=2, ls='--', label=f'Median = {sojourns_min.median():.1f} min')
axes[0].axvline(sojourns_min.quantile(0.95), color='green', lw=2, ls=':', label=f'95th pct = {sojourns_min.quantile(0.95):.1f} min')
axes[0].set_xlabel('Time in clinic (min)')
axes[0].set_ylabel('Count')
axes[0].set_title('Sojourn time distribution (c_nurse=2)')
axes[0].legend(fontsize=8)

# ECDF
sorted_s = np.sort(sojourns_min)
ecdf_y = np.arange(1, len(sorted_s)+1) / len(sorted_s)
axes[1].plot(sorted_s, ecdf_y, color='steelblue', lw=1.5)
axes[1].axhline(0.90, color='gray', ls='--', lw=1, label='90th percentile')
idx_90 = np.searchsorted(ecdf_y, 0.90)
axes[1].axvline(sorted_s[idx_90], color='gray', ls='--', lw=1)
axes[1].set_xlabel('Time in clinic (min)')
axes[1].set_ylabel('P(sojourn ≤ t)')
axes[1].set_title('ECDF of sojourn time')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Try It Yourself

1. **Throughput target**: The clinic wants to serve 50 patients in an 8-hour day with P(sojourn > 90 min) ≤ 0.05. Use simulation to find the minimum staffing configuration (c_reg, c_nurse, c_exam) that meets this target.

2. **Routing fraction**: Change the model so that only 60% of patients need a triage nurse (40% go directly from registration to the physician). Update the `patient()` process with a random routing branch. How does the bottleneck shift?

3. **Jackson networks**: Look up Jackson's theorem, which states that an open queueing network with Poisson arrivals and exponential service has product-form steady-state distribution. Verify whether the simulation results are consistent with the product-form prediction for the three-stage clinic. (Hint: treat inter-stage flows as Poisson and apply M/M/c to each stage — does the sum match simulation?)